# RQ4: does the distribution of UMDA4GI help LLM-guided search?

**Definitions** (the same in every notebook, see the docstring of `analysis.py`):

- *improvement*: % change in CPU cycles of the returned patch on the validation inputs (negative = faster).
  A patch that fails the correctness oracle on the validation inputs counts as 0% (the original program is kept).
- *gap*: for each run, validation change minus training change, in percentage points (positive = the patch
  generalises worse than it trained). Aggregated with the median of the runs.
- *cell*: (software, algorithm, budget), 15 runs = 5 folds × 3 seeds. Each run has 1000 evaluations; the budgets
  100, 250, 500 and 750 are checkpoints of the same run.
- *ranking*: per software, the median of each cell, averaged over the five budgets and ranked; rank sum over the
  four systems (lower = better).

Groups: UMDA4GI without LLM, UMDA4GI whose population is proposed by an LLM (`UMDA-qwen7b`, `UMDA-llama8b`) and the LLM-only algorithm (`LLM-qwen7b`, `LLM-llama8b`). The LLMs are qwen2.5-coder:7b and llama3.1:8b, served locally with Ollama.

In [1]:
import pathlib
import sys

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(pathlib.Path('..', 'revision_analysis').resolve()))
import analysis as A

RESULTS = 'results.csv'           # one row per run and budget (checkpoint)
OUT = pathlib.Path('analysis')    # tables written by analysis.py
REFRESH = False                   # True: recompute every table from results.csv
if REFRESH or not (OUT / 'cells.csv').exists():
    A.run(RESULTS, OUT, 'zero')
df = A.load(RESULTS, 'zero')
cells = pd.read_csv(OUT / 'cells.csv')
SW = {'minisat_hack': 'MiniSAT-hack', 'sat4j': 'Sat4J', 'weka': 'Weka', 'pngOptim': 'OptiPNG'}
pd.set_option('display.precision', 2)
pd.set_option('display.max_rows', 300)
pd.set_option('display.width', 200)
G = A.RQ4
print('LLM runs:', df[df.group.isin(A.LLM)].id.nunique())

LLM runs: 240


## Median improvement and gap per software, group and budget

Median of the 15 runs with its bootstrap 95% confidence interval.

In [2]:
c = cells[cells.group.isin(G)].copy()
for m in ['improvement', 'gap']:
    c['value'] = c.apply(lambda r: f"{r[m + '_median']:.2f} [{r[m + '_ci_low']:.2f}, {r[m + '_ci_high']:.2f}]", axis=1)
    print(m)
    display(c.pivot_table(index=['software', 'group'], columns='budget', values='value', aggfunc='first'))

improvement


budget                                     100                    250                   500                     750                    1000
software     group                                                                                                                         
minisat_hack LLM-llama8b     0.25 [-5.98, 9.98]  -1.77 [-11.90, 13.69]  9.98 [-32.24, 60.68]  -16.46 [-38.05, 14.08]   1.50 [-38.05, 18.21]
             LLM-qwen7b     -0.35 [-2.50, 9.43]     3.02 [0.00, 55.31]   19.74 [0.00, 55.81]     20.01 [0.00, 55.81]    13.03 [0.00, 55.81]
             UMDA           -0.56 [-3.21, 2.99]     0.61 [-4.00, 2.99]    0.35 [-4.00, 5.44]     -0.68 [-4.00, 5.61]  -0.39 [-20.14, 11.74]
             UMDA-llama8b     5.33 [0.22, 7.27]    3.34 [-3.80, 20.43]   0.19 [-3.33, 19.35]    1.85 [-22.30, 14.99]  -2.39 [-18.99, 14.99]
             UMDA-qwen7b   0.72 [-21.81, 13.17]     5.27 [0.00, 20.03]   1.34 [-2.12, 20.81]     0.00 [-8.61, 15.19]    6.13 [-9.00, 15.17]
pngOptim     LLM-llama8b    -0.02 [-0.14, 0.19]     0.04 [-0.06, 0.22]   -0.06 [-0.98, 0.04]     -0.01 [-0.09, 0.33]   -0.03 [-15.08, 0.01]
             LLM-qwen7b     -0.02 [-0.11, 0.20]    -0.03 [-0.14, 0.19]   -0.09 [-0.16, 0.01]     -0.07 [-0.16, 0.01]    -0.02 [-0.07, 0.04]
             UMDA            0.00 [-0.12, 0.09]    -0.02 [-0.14, 0.09]   -0.02 [-0.12, 0.02]     -0.03 [-0.14, 0.08]    -0.12 [-0.14, 0.00]
             UMDA-llama8b   -0.03 [-0.30, 0.13]    -0.01 [-0.08, 0.11]    0.00 [-0.03, 0.11]      0.00 [-0.04, 0.13]     0.00 [-0.03, 0.11]
             UMDA-qwen7b      0.03 [0.01, 0.08]     0.00 [-0.04, 0.02]    0.00 [-0.03, 0.03]      0.02 [-0.27, 0.15]    -0.06 [-0.14, 0.09]
sat4j        LLM-llama8b     0.00 [-3.31, 1.45]    -1.05 [-2.86, 1.89]    1.36 [-3.29, 5.98]      0.65 [-3.49, 5.90]    -0.89 [-2.71, 2.04]
             LLM-qwen7b     -0.31 [-2.07, 2.26]     0.95 [-2.07, 2.26]    0.52 [-2.75, 2.11]     -0.01 [-2.75, 1.34]    -0.53 [-2.75, 1.19]
             UMDA           -0.70 [-1.04, 0.64]    -1.16 [-2.88, 0.33]   -0.77 [-2.88, 0.33]     -0.22 [-2.88, 0.93]     0.04 [-2.88, 1.60]
             UMDA-llama8b  -1.34 [-1.84, -0.32]   -1.69 [-2.21, -0.89]   -0.43 [-2.21, 3.36]     -0.35 [-3.20, 0.71]    -0.03 [-2.93, 0.90]
             UMDA-qwen7b     0.45 [-1.81, 0.95]     0.58 [-0.05, 4.43]   -0.34 [-2.44, 3.65]      1.62 [-0.94, 3.65]    -0.22 [-3.51, 4.11]
weka         LLM-llama8b      0.14 [0.00, 1.33]     0.07 [-0.53, 0.76]    0.03 [-1.04, 0.76]     -0.16 [-1.04, 0.76]    -0.19 [-0.82, 0.49]
             LLM-qwen7b     -0.61 [-1.77, 1.87]     0.22 [-2.25, 1.87]    0.15 [-1.10, 0.71]     -0.22 [-1.58, 0.65]    -0.22 [-1.03, 0.65]
             UMDA          -0.64 [-1.16, -0.08]   -0.50 [-1.16, -0.08]  -0.64 [-1.56, -0.30]    -0.64 [-1.16, -0.30]    -0.66 [-3.07, 0.01]
             UMDA-llama8b    0.36 [-1.12, 1.60]     0.54 [-1.12, 2.75]   -0.95 [-2.30, 0.54]      0.14 [-1.81, 0.60]     1.11 [-0.16, 2.31]
             UMDA-qwen7b    -0.53 [-1.86, 2.15]     0.12 [-1.86, 2.15]    0.14 [-1.82, 1.35]      0.12 [-2.28, 0.90]     0.10 [-1.58, 0.82]

gap


budget                                    100                   250                   500                   750                   1000
software     group                                                                                                                    
minisat_hack LLM-llama8b   11.46 [2.00, 34.55]  21.93 [-3.24, 44.96]  46.22 [12.76, 86.84]  17.58 [-8.30, 54.44]  38.30 [-3.72, 54.44]
             LLM-qwen7b    12.37 [0.05, 19.92]  59.72 [19.65, 78.12]  78.09 [24.30, 84.42]  78.09 [55.71, 80.68]  74.28 [21.35, 79.65]
             UMDA          2.87 [-1.07, 19.24]    9.56 [1.76, 19.22]    9.56 [0.61, 49.12]   12.08 [0.86, 20.49]  21.23 [-0.57, 48.78]
             UMDA-llama8b  12.84 [3.95, 31.78]  21.83 [11.38, 56.34]   24.07 [4.29, 56.72]   20.51 [4.90, 51.72]   20.51 [4.68, 59.22]
             UMDA-qwen7b   12.32 [1.54, 21.47]  36.09 [20.37, 78.06]  36.09 [11.83, 58.17]   40.65 [2.65, 57.81]   41.28 [4.65, 58.36]
pngOptim     LLM-llama8b     0.13 [0.02, 0.30]    0.24 [-0.01, 0.39]    0.12 [-0.03, 0.50]    0.19 [-0.03, 0.55]    0.22 [-0.10, 0.60]
             LLM-qwen7b     0.03 [-0.03, 0.41]    0.10 [-0.04, 0.23]    0.10 [-0.04, 0.35]    0.10 [-0.04, 0.31]     0.19 [0.11, 0.24]
             UMDA            0.07 [0.00, 0.16]    0.00 [-0.08, 0.12]    0.00 [-0.08, 0.12]    0.00 [-0.10, 0.12]   -0.03 [-0.22, 0.08]
             UMDA-llama8b   0.09 [-0.23, 0.23]    0.11 [-0.00, 0.25]     0.18 [0.09, 0.25]     0.15 [0.00, 0.25]     0.12 [0.09, 0.25]
             UMDA-qwen7b     0.24 [0.14, 0.29]     0.17 [0.11, 0.25]     0.23 [0.15, 0.36]     0.19 [0.15, 0.41]     0.26 [0.18, 0.41]
sat4j        LLM-llama8b    2.32 [-0.01, 6.91]    4.70 [-0.61, 8.03]    6.68 [4.67, 26.65]   13.62 [3.74, 23.29]   14.38 [5.42, 23.29]
             LLM-qwen7b      4.57 [3.02, 9.28]    7.35 [3.02, 12.66]    9.28 [5.68, 14.06]   12.66 [5.20, 18.63]   14.06 [9.52, 27.58]
             UMDA            5.64 [2.46, 7.72]     5.64 [2.46, 7.52]    5.90 [2.96, 11.09]    9.91 [5.64, 12.11]    9.91 [5.89, 16.47]
             UMDA-llama8b    3.78 [1.48, 4.70]     3.96 [1.10, 4.91]    6.88 [4.28, 17.68]    9.06 [4.64, 12.51]    9.06 [5.91, 15.67]
             UMDA-qwen7b     5.13 [2.76, 5.99]    5.97 [4.56, 12.95]    6.79 [3.11, 22.07]   12.95 [9.64, 23.23]   19.38 [6.33, 26.01]
weka         LLM-llama8b     2.34 [0.47, 3.21]     2.54 [1.31, 3.76]     2.34 [1.31, 3.89]     1.93 [1.31, 3.76]     1.93 [1.08, 3.89]
             LLM-qwen7b      1.29 [0.00, 3.94]    2.12 [-0.46, 4.06]     2.23 [0.29, 3.21]    2.12 [-0.19, 3.96]     2.03 [0.47, 4.06]
             UMDA            1.25 [0.09, 3.67]     1.25 [0.44, 3.23]     0.87 [0.09, 2.61]     0.87 [0.00, 2.61]    0.44 [-0.37, 3.93]
             UMDA-llama8b    1.78 [0.54, 3.88]     2.55 [0.89, 4.25]     1.23 [0.89, 3.64]     2.66 [0.91, 4.46]     4.04 [2.33, 5.33]
             UMDA-qwen7b    0.91 [-0.47, 3.61]     1.64 [0.01, 4.37]     1.94 [0.37, 4.88]     3.00 [0.37, 4.17]     3.07 [2.12, 4.35]

## Rankings

In [3]:
display(pd.read_csv(OUT / 'rq4_improvement_ranking.csv'))
display(pd.read_csv(OUT / 'rq4_gap_ranking.csv'))

,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,-0.13,-3.65e-02,-0.56,-0.62,2.0,2.0,2.0,1.0,7.0
1,LLM-llama8b,-1.30,-1.60e-02,0.01,-0.02,1.0,3.0,3.0,3.0,10.0
2,LLM-qwen7b,11.09,-4.59e-02,0.12,-0.14,5.0,1.0,4.0,2.0,12.0
3,UMDA-llama8b,1.66,-9.26e-03,-0.77,0.24,3.0,4.0,1.0,5.0,13.0
4,UMDA-qwen7b,2.69,-2.98e-03,0.42,-0.01,4.0,5.0,5.0,4.0,18.0


,group,minisat_hack_value,pngOptim_value,sat4j_value,weka_value,minisat_hack_rank,pngOptim_rank,sat4j_rank,weka_rank,rank_sum
0,UMDA,11.06,8.06e-03,7.40,0.94,1.0,1.0,2.0,1.0,5.0
1,UMDA-llama8b,19.95,1.27e-01,6.55,2.45,2.0,3.0,1.0,5.0,11.0
2,LLM-qwen7b,60.51,1.02e-01,9.59,1.96,5.0,2.0,4.0,2.0,13.0
3,LLM-llama8b,27.10,1.81e-01,8.34,2.22,3.0,4.0,3.0,4.0,14.0
4,UMDA-qwen7b,33.28,2.19e-01,10.04,2.11,4.0,5.0,5.0,3.0,17.0


## Friedman and pairwise tests

Friedman over the five groups, with the 15 (fold, seed) pairs as blocks; then Wilcoxon signed-rank tests with Holm's correction within each (software, budget, metric), i.e. over its 10 pairs.

In [4]:
fr = pd.read_csv(OUT / 'rq4_friedman.csv')
display(fr.pivot_table(index=['metric', 'software'], columns='budget', values='p'))
print('Friedman p < 0.05:', fr[fr.p < 0.05].groupby('metric').size().to_dict(), 'of 20 cells per metric')
pw = pd.read_csv(OUT / 'rq4_pairwise.csv')
s = pw[(pw.p_holm < 0.05) & pw.metric.isin(['improvement', 'gap'])]
display(s)
u = s[(s.group_a == 'UMDA') | (s.group_b == 'UMDA')]
umda_better = ((u.group_a == 'UMDA') & (u.median_a < u.median_b)) | ((u.group_b == 'UMDA') & (u.median_b < u.median_a))
print(f'significant comparisons with UMDA4GI: {len(u)}, in favour of UMDA4GI: {umda_better.sum()}')

budget                    100       250       500       750       1000
metric      software                                                  
gap         minisat_hack  0.07  9.10e-03  4.01e-03  1.12e-02  7.71e-02
            pngOptim      0.06  5.33e-02  3.83e-03  1.38e-02  9.10e-03
            sat4j         0.19  1.05e-02  9.38e-01  3.20e-01  2.70e-01
            weka          0.57  1.64e-01  3.51e-01  5.57e-02  1.61e-01
improvement minisat_hack  0.09  3.17e-02  1.62e-01  1.84e-02  1.71e-01
            pngOptim      0.53  7.41e-01  3.58e-02  5.51e-01  8.28e-01
            sat4j         0.27  5.57e-02  1.34e-01  5.69e-01  9.52e-01
            weka          0.30  6.43e-01  6.72e-01  7.11e-01  3.26e-01
train       minisat_hack  0.78  2.08e-02  9.97e-05  7.79e-05  7.03e-06
            pngOptim      0.06  1.35e-02  2.81e-03  2.02e-03  6.36e-04
            sat4j         0.07  1.39e-01  7.80e-01  8.56e-01  4.99e-01
            weka          0.53  2.55e-01  1.06e-01  6.35e-02  1.55e-01

Friedman p < 0.05: {'gap': 7, 'improvement': 3, 'train': 8} of 20 cells per metric


,software,budget,group_a,group_b,metric,blocks,median_a,median_b,a12_a_better,p,p_holm
1,minisat_hack,100,UMDA,UMDA-llama8b,improvement,15,-0.56,5.33,0.73,4.27e-03,4.27e-02
42,minisat_hack,250,UMDA,LLM-qwen7b,gap,15,9.56,59.72,0.78,2.62e-03,2.62e-02
71,minisat_hack,500,UMDA,UMDA-llama8b,gap,15,9.56,24.07,0.64,8.54e-04,8.54e-03
72,minisat_hack,500,UMDA,LLM-qwen7b,gap,15,9.56,78.09,0.79,5.37e-03,4.83e-02
160,pngOptim,100,UMDA,UMDA-qwen7b,gap,15,0.07,0.24,0.82,8.54e-04,8.54e-03
190,pngOptim,250,UMDA,UMDA-qwen7b,gap,15,0.00,0.17,0.81,6.10e-04,6.10e-03
218,pngOptim,500,UMDA-llama8b,LLM-llama8b,improvement,15,0.00,-0.06,0.33,4.27e-03,4.27e-02
220,pngOptim,500,UMDA,UMDA-qwen7b,gap,15,0.00,0.23,0.89,6.10e-05,6.10e-04
221,pngOptim,500,UMDA,UMDA-llama8b,gap,15,0.00,0.18,0.78,3.36e-03,3.02e-02
344,sat4j,250,UMDA-qwen7b,UMDA-llama8b,gap,15,5.97,3.96,0.21,2.62e-03,2.62e-02


significant comparisons with UMDA4GI: 9, in favour of UMDA4GI: 9


## Training against validation

The LLM groups improve the training inputs more than UMDA4GI, but not the validation inputs, so their gap is larger. The tests on the training change (`metric = train`) were added after seeing the results and are exploratory. Medians at 1000 evaluations:

In [5]:
last = cells[cells.group.isin(G) & (cells.budget == 1000)]
display(last.pivot(index='software', columns='group', values='train_median')[G].rename_axis(columns='training'))
display(last.pivot(index='software', columns='group', values='improvement_median')[G].rename_axis(columns='validation'))
display(last.pivot(index='software', columns='group', values='gap_median')[G].rename_axis(columns='gap'))

training,UMDA,UMDA-qwen7b,UMDA-llama8b,LLM-qwen7b,LLM-llama8b
software,,,,,
minisat_hack,-23.21,-33.51,-35.51,-36.02,-40.39
pngOptim,-0.07,-0.27,-0.13,-0.14,-0.49
sat4j,-8.71,-10.83,-7.98,-13.95,-14.81
weka,-1.89,-2.83,-3.49,-2.23,-2.27


validation,UMDA,UMDA-qwen7b,UMDA-llama8b,LLM-qwen7b,LLM-llama8b
software,,,,,
minisat_hack,-0.39,6.13,-2.39,13.03,1.50
pngOptim,-0.12,-0.06,0.00,-0.02,-0.03
sat4j,0.04,-0.22,-0.03,-0.53,-0.89
weka,-0.66,0.10,1.11,-0.22,-0.19


gap,UMDA,UMDA-qwen7b,UMDA-llama8b,LLM-qwen7b,LLM-llama8b
software,,,,,
minisat_hack,21.23,41.28,20.51,74.28,38.30
pngOptim,-0.03,0.26,0.12,0.19,0.22
sat4j,9.91,19.38,9.06,14.06,14.38
weka,0.44,3.07,4.04,2.03,1.93


Each LLM group against UMDA4GI, per metric: in how many of the 20 cells (software × budget) it is significantly better or worse (Holm within the cell).

In [6]:
rows = []
for g in A.LLM:
    for m in ['train', 'improvement', 'gap']:
        t = pw[(pw.metric == m) & pw.group_a.isin([g, 'UMDA']) & pw.group_b.isin([g, 'UMDA'])]
        sig = t[t.p_holm < 0.05]
        g_med = np.where(sig.group_a == g, sig.median_a, sig.median_b)
        u_med = np.where(sig.group_a == g, sig.median_b, sig.median_a)
        rows.append({'group': g, 'metric': m, 'cells': len(t),
                     'LLM group better': int((g_med < u_med).sum()), 'UMDA4GI better': int((g_med > u_med).sum())})
display(pd.DataFrame(rows).pivot(index='group', columns='metric', values=['LLM group better', 'UMDA4GI better']))

LLM group better                   UMDA4GI better                  
metric                    gap improvement train            gap improvement train
group                                                                           
LLM-llama8b                 0           0     9              0           0     2
LLM-qwen7b                  0           0     4              2           0     0
UMDA-llama8b                0           0     3              3           1     0
UMDA-qwen7b                 0           0     7              3           0     0

## Does the hybrid overfit less than the LLM alone?

Each UMDA4GI+LLM group against the LLM-only algorithm with the same model.

In [7]:
for hybrid, alone in [('UMDA-llama8b', 'LLM-llama8b'), ('UMDA-qwen7b', 'LLM-qwen7b')]:
    for m in ['gap', 'train', 'improvement']:
        t = pw[(pw.metric == m) & pw.group_a.isin([hybrid, alone]) & pw.group_b.isin([hybrid, alone])]
        h_med = np.where(t.group_a == hybrid, t.median_a, t.median_b)
        a_med = np.where(t.group_a == hybrid, t.median_b, t.median_a)
        sig = t.p_holm < 0.05
        print(f'{hybrid} vs {alone}, {m}: lower median in {(h_med < a_med).sum()}/20 cells; '
              f'significant: {int((sig & (h_med < a_med)).sum())} for the hybrid, {int((sig & (h_med > a_med)).sum())} for the LLM alone')

UMDA-llama8b vs LLM-llama8b, gap: lower median in 12/20 cells; significant: 0 for the hybrid, 1 for the LLM alone
UMDA-llama8b vs LLM-llama8b, train: lower median in 9/20 cells; significant: 0 for the hybrid, 3 for the LLM alone
UMDA-llama8b vs LLM-llama8b, improvement: lower median in 9/20 cells; significant: 0 for the hybrid, 1 for the LLM alone
UMDA-qwen7b vs LLM-qwen7b, gap: lower median in 10/20 cells; significant: 0 for the hybrid, 0 for the LLM alone
UMDA-qwen7b vs LLM-qwen7b, train: lower median in 10/20 cells; significant: 0 for the hybrid, 0 for the LLM alone
UMDA-qwen7b vs LLM-qwen7b, improvement: lower median in 8/20 cells; significant: 0 for the hybrid, 0 for the LLM alone


## Patch length does not explain it

The ablations show that the small gap of UMDA4GI goes with its short patches. The LLM-only patches are as short or shorter, and still generalise worse: mean number of edits at 1000 evaluations.

In [8]:
display(last.pivot(index='software', columns='group', values='patch_length_mean')[G])

group,UMDA,UMDA-qwen7b,UMDA-llama8b,LLM-qwen7b,LLM-llama8b
software,,,,,
minisat_hack,1.80,1.80,1.87,1.60,1.93
pngOptim,1.60,1.40,1.40,1.33,1.53
sat4j,1.67,1.40,1.60,1.67,1.60
weka,1.60,1.47,1.73,1.00,1.07


## Per fold

Median gap of the 3 seeds of each fold at 1000 evaluations. On MiniSAT the large gaps of the LLM groups come mostly from fold 1; on Weka the pattern holds in every fold.

In [9]:
d = df[df.group.isin(G) & (df.checkpoint == 1000) & df.software.isin(['minisat_hack', 'sat4j', 'weka'])]
display(d.pivot_table(index=['software', 'group'], columns='fold', values='gap', aggfunc='median').round(1))

fold                           1     2     3     4     5
software     group                                      
minisat_hack LLM-llama8b   158.1  49.9  45.8 -37.1  -3.7
             LLM-qwen7b    160.2  -8.7  62.1  80.7  78.1
             UMDA           21.2  22.9  48.8 -63.9 -10.0
             UMDA-llama8b  193.5  20.5  51.6 -40.9   4.7
             UMDA-qwen7b    47.6  41.3  57.8 -66.8   4.7
sat4j        LLM-llama8b    14.3  23.3  24.0  13.5   2.4
             LLM-qwen7b     23.6  14.1  29.0   6.3   9.6
             UMDA           20.6   8.6  16.5   8.1   5.6
             UMDA-llama8b   17.7   7.5   8.8   5.9   9.1
             UMDA-qwen7b    23.2  13.8  45.4  12.9   3.9
weka         LLM-llama8b     1.2   3.8   1.8   3.9   2.5
             LLM-qwen7b      4.0   2.6   1.5   2.0   1.0
             UMDA            0.1   0.0   0.6   4.2   0.4
             UMDA-llama8b    3.3   3.0   3.6   4.5   6.7
             UMDA-qwen7b     2.3   4.9   1.2   4.3   2.2

## Patches that fail the correctness oracle on the validation inputs

They count as 0% improvement. All are on MiniSAT fold 5: the patch cuts about 78% of the training cycles and the program fails on the validation inputs.

In [10]:
f = df[df.val_failed & df.group.isin(G)]
display(f[['software', 'group', 'id', 'fold', 'seed', 'checkpoint', 'change_perc_train', 'validate_status']])

,software,group,id,fold,seed,checkpoint,change_perc_train,validate_status
346,minisat_hack,UMDA-qwen7b,70,5,105,250,-78.06,RUN_CODE_ERROR
347,minisat_hack,UMDA-qwen7b,70,5,105,500,-78.06,RUN_CODE_ERROR
348,minisat_hack,UMDA-qwen7b,70,5,105,750,-79.16,RUN_CODE_ERROR
349,minisat_hack,UMDA-qwen7b,70,5,105,1000,-79.16,RUN_CODE_ERROR
471,minisat_hack,LLM-qwen7b,95,5,5,250,-78.09,RUN_CODE_ERROR
472,minisat_hack,LLM-qwen7b,95,5,5,500,-78.09,RUN_CODE_ERROR
473,minisat_hack,LLM-qwen7b,95,5,5,750,-78.09,RUN_CODE_ERROR
474,minisat_hack,LLM-qwen7b,95,5,5,1000,-78.09,RUN_CODE_ERROR
496,minisat_hack,LLM-qwen7b,100,5,105,250,-77.76,RUN_CODE_ERROR
497,minisat_hack,LLM-qwen7b,100,5,105,500,-78.56,RUN_CODE_ERROR


## How much of the LLM output was used (Reviewer 2, comment 3)

Per group and software, share of the patches asked to the LLM that were malformed, missing (the LLM returned too few) or duplicated (well-formed but already in the population, or empty). All three are replaced by random one-edit patches, in UMDA4GI+LLM and in the LLM-only algorithm. `fallback_rate` = malformed + missing; `not_used_rate` adds the duplicates. Also the time per LLM call.

In [11]:
fb = pd.read_csv(OUT / 'rq4_llm_fallback.csv')
display(fb)
display(fb.groupby('group')[['fallback_rate', 'not_used_rate']].agg(['min', 'max']))

,group,software,runs,llm_requested,llm_malformed,llm_missing,llm_duplicate,llm_calls,llm_minutes,malformed_rate,missing_rate,duplicate_rate,fallback_rate,not_used_rate,seconds_per_call
0,LLM-llama8b,minisat_hack,15,13965.0,7626.0,0.0,3273.0,285,2770.60,5.46e-01,0.00e+00,0.23,5.46e-01,0.78,583.28
1,LLM-llama8b,pngOptim,15,13965.0,4398.0,41.0,5385.0,285,1263.23,3.15e-01,2.94e-03,0.39,3.18e-01,0.70,265.94
2,LLM-llama8b,sat4j,15,13965.0,1648.0,1.0,7687.0,285,4898.55,1.18e-01,7.16e-05,0.55,1.18e-01,0.67,1031.27
3,LLM-llama8b,weka,15,13965.0,4241.0,0.0,4866.0,285,1685.88,3.04e-01,0.00e+00,0.35,3.04e-01,0.65,354.92
4,LLM-qwen7b,minisat_hack,15,13524.0,4407.0,690.0,3019.0,276,1096.41,3.26e-01,5.10e-02,0.22,3.77e-01,0.60,238.35
5,LLM-qwen7b,pngOptim,15,13965.0,7457.0,855.0,1940.0,285,2475.18,5.34e-01,6.12e-02,0.14,5.95e-01,0.73,521.09
6,LLM-qwen7b,sat4j,15,13965.0,2695.0,1950.0,2813.0,285,3569.14,1.93e-01,1.40e-01,0.20,3.33e-01,0.53,751.40
7,LLM-qwen7b,weka,15,13965.0,5527.0,1478.0,1974.0,285,4641.29,3.96e-01,1.06e-01,0.14,5.02e-01,0.64,977.11
8,UMDA-llama8b,minisat_hack,15,13965.0,625.0,5.0,6480.0,285,432.45,4.48e-02,3.58e-04,0.46,4.51e-02,0.51,91.04
9,UMDA-llama8b,pngOptim,15,13965.0,1176.0,0.0,5570.0,285,961.78,8.42e-02,0.00e+00,0.40,8.42e-02,0.48,202.48


fallback_rate       not_used_rate      
                       min   max           min   max
group                                               
LLM-llama8b       1.18e-01  0.55          0.65  0.78
LLM-qwen7b        3.33e-01  0.60          0.53  0.73
UMDA-llama8b      9.31e-03  0.08          0.43  0.51
UMDA-qwen7b       1.26e-01  0.29          0.55  0.65

## Diversity of the returned patches

Distinct edited locations over total edits of the patches returned at 1000 evaluations, per software and pooled over the four systems.

In [12]:
dv = pd.read_csv(OUT / 'diversity.csv')
dv = dv[dv.group.isin(G)]
display(dv)
t = dv.groupby('group')[['distinct_locations', 'edits']].sum()
display((t.distinct_locations / t.edits).rename('pooled diversity').sort_values(ascending=False))

,software,group,edits,distinct_locations,diversity
2,minisat_hack,LLM-llama8b,29,21,0.72
3,minisat_hack,LLM-qwen7b,24,20,0.83
5,minisat_hack,UMDA,27,22,0.81
6,minisat_hack,UMDA-llama8b,28,21,0.75
7,minisat_hack,UMDA-qwen7b,27,22,0.81
10,pngOptim,LLM-llama8b,23,18,0.78
11,pngOptim,LLM-qwen7b,20,20,1.00
13,pngOptim,UMDA,24,24,1.00
14,pngOptim,UMDA-llama8b,21,21,1.00
15,pngOptim,UMDA-qwen7b,21,19,0.90


group
LLM-qwen7b      0.90
UMDA            0.90
UMDA-llama8b    0.90
UMDA-qwen7b     0.86
LLM-llama8b     0.79
Name: pooled diversity, dtype: float64